## Hyperparameter Tuning with Optuna

This notebook performs hyperparameter optimization for the selected models
(**CatBoost**, **XGBoost**, and **Extra Trees**) using Optuna.

The optimization objective is **Average Precision (PR-AUC)**, which is more
appropriate for highly imbalanced classification problems such as fraud detection.

Sampling techniques and threshold tuning are intentionally excluded at this stage
to isolate the intrinsic performance of each model.

## Objectives

* Tune the hyperparameters of the three best-performing models identified in the previous step
* Search for hyperparameter configurations that improve **Average Precision**
* Select the best hyperparameters for each model
* Evaluate model performance using metrics suitable for imbalanced classification
* Compare the models using a consistent evaluation framework

## Configuration

All project configurations are centralized in the **config.yaml** file located in the project root directory.

This file contains parameters related to:

* Data paths
* Model configurations
* Experiment settings

The notebook loads these parameters to ensure reproducibility and consistency across experiments.

## Data Source

This notebook depends on the output generated by the notebook:

```
3-Feature-Engineering.ipynb
```

If that notebook has not been executed previously, the required dataset will not exist.

The training dataset used in this stage is located at:

```
../data/splits/feature_engineered/train_fe.parquet
```

This path is relative to the notebook location within the project structure.

## Optimization Strategy

Hyperparameter optimization is performed using **Optuna**, a framework for efficient hyperparameter search.

The optimization process follows these principles:

* Each model is optimized independently.
* The objective metric is **Average Precision (PR-AUC)**.
* Hyperparameters are sampled from predefined search spaces.
* Models are trained using **StratifiedKFold** cross-validation with **5 splits**.
* The number of trials is set to **50**.
* Model performance is evaluated using a consistent validation strategy.

Optuna explores the search space using an adaptive sampling strategy to efficiently identify promising hyperparameter configurations.


## Models Evaluated

The following algorithms are evaluated:

* Extra Trees
* XGBoost
* CatBoost

## Research vs Production Code

This notebook was created during the research and experimentation phase of the project.

While it contains exploratory implementations, the **production-ready machine learning pipeline** is implemented in:

```
src/datapipeline
```

This ensures that the final workflow used in production is **modular, testable, and reproducible**.


In [ ]:
from datapipeline.training.load_data import load_raw_data
from datapipeline.config.mlflow_config import setup_mlflow
import yaml
import pandas as pd
import mlflow
from pathlib import Path
import tempfile
import optuna
import yaml
from sklearn.metrics import (roc_auc_score, 
                            balanced_accuracy_score,
                            precision_score,
                            recall_score,
                            f1_score,
                            confusion_matrix,
                            precision_recall_curve,
                            average_precision_score,
                            make_scorer)
from sklearn.model_selection import cross_validate
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from lightgbm import LGBMClassifier
import xgboost as xgb
from sklearn.utils.class_weight import compute_sample_weight


In [ ]:
#primary metric that will be used for model comparison
PRIMARY_METRIC = "Average Precision Score"


In [ ]:
#loading config file
config_path = '../config.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

In [ ]:
random_state = config["parameters"]["random_state"]
artifacts_path = config['5-hyperparameter_tuning']['artifacts_path']

# MLflow

In [ ]:
mlflow_local_folder = config['mlflow']['experiment_path']

In [ ]:
experiment_name = config['pipeline']['experiment_name']


In [ ]:
setup_mlflow(experiment_name, mlflow_local_folder)

In [ ]:
mlflow.start_run(run_name='Hyperparameter Tuning')

In [ ]:
mlflow.set_tag('Type', 'Research')
mlflow.set_tag('Notebook', '5-Hyperparameter Tuning')

# Load Dataset

In [ ]:
dataset_path = Path(config['datasets']['train_feature_engineered_path'])
target_column = config['parameters']['target_column']

In [ ]:
df_train = pd.read_parquet(dataset_path)

In [ ]:
y_train = df_train[target_column]

In [ ]:
x_train = df_train.drop(columns = ['Time', target_column])

# Models

Models selected in the previous step:

- Catboost
- Extra Trees
- Xgboost

# Cross Validation setup

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=config["model_training"]["random_state"]
)

scoring = {
    'average_precision': 'average_precision'
          }

# Optuna objective — XGBoost

In [ ]:
n_major = sum(y_train==0)
n_minor = sum(y_train==1)
scale_pos_weight = n_major / n_minor

In [ ]:
def calculating_cross_validation(model):
    cv_results = cross_validate(model, 
                                x_train, 
                                y_train, 
                                cv=cv, 
                                scoring=scoring, 
                                verbose=False)
    return cv_results
    

In [ ]:
def objective_xgboost(trial):

     params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 20.0),
        "random_state": random_state,
        "scale_pos_weight": scale_pos_weight,
        "eval_metric": "aucpr",
        "tree_method": "hist"
     }

     model = xgb.XGBClassifier(**params)

     '''
     cv_results = cross_validate(model, 
                                x_train, 
                                y_train, 
                                cv=cv, 
                                scoring=scoring, 
                                verbose=False)
     '''
     cv_results = calculating_cross_validation(model)
     metrics_results = [cv_results[f'test_{metric}'].mean() for metric in scoring.keys()]

     return metrics_results

In [ ]:
with mlflow.start_run(run_name="xgboost_hyperparameter_tuning", nested = True):
    mlflow.set_tag('model_name', 'xgboost')
    study = optuna.create_study(direction='maximize')
    study.optimize(objective_xgboost, n_trials=50)

    model = xgb.XGBClassifier(scale_pos_weight=scale_pos_weight)
    cv_results = calculating_cross_validation(model)
    xgb_best_params = study.best_params
    metric_default_hyperparameters = [cv_results[f'test_{metric}'].mean() for metric in scoring.keys()]

    mlflow.log_metric('average_precision_default_params', metric_default_hyperparameters[0])
    mlflow.log_metric('best_cv_average_precision', study.best_value)
    mlflow.log_params(study.best_params)

print(f'Best average precision score: {study.best_value}')
print(f'Best parameters: {study.best_params}')


# Optuna objective — CatBoost

In [ ]:
def objective_catboost(trial):

    params = {
        "iterations": trial.suggest_int("iterations", 300, 800),
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "random_state": random_state,
        "verbose": False
    }

    model = CatBoostClassifier(**params)

    cv_results = calculating_cross_validation(model)
    metrics_results = [cv_results[f'test_{metric}'].mean() for metric in scoring.keys()]

    return metrics_results

In [ ]:
with mlflow.start_run(run_name="Catboost_hyperparameter_tuning", nested = True):
    mlflow.set_tag('model_name', 'catboost')
    study = optuna.create_study(direction='maximize')
    study.optimize(objective_catboost, n_trials=50)

    model = CatBoostClassifier()
    cv_results = calculating_cross_validation(model)
    metric_default_hyperparameters = [cv_results[f'test_{metric}'].mean() for metric in scoring.keys()]
    catboost_best_params = study.best_params

    mlflow.log_metric('average_precision_default_params', metric_default_hyperparameters[0])
    mlflow.log_metric('best_cv_average_precision', study.best_value)
    mlflow.log_params(study.best_params)
    
print(f'Best average precision score: {study.best_params}')
print(f'Best parameters: {study.best_params}')

# Optuna objective — Extra Trees

In [ ]:
def objective_extra_trees(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 800),
        "max_depth": trial.suggest_int("max_depth", 5, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_float("max_features", 0.3, 1.0),
        "class_weight": "balanced",
        "random_state": config["model_training"]["random_state"],
        "n_jobs": -1
    }

    model = ExtraTreesClassifier(**params)

    cv_results = calculating_cross_validation(model)

    metrics_results = [cv_results[f'test_{metric}'].mean() for metric in scoring.keys()]

    return metrics_results

In [ ]:
with mlflow.start_run(run_name="ExtraTrees_hyperparameter_tuning", nested = True):
    mlflow.set_tag('model_name', 'extratrees')
    study = optuna.create_study(direction='maximize')
    study.optimize(objective_extra_trees, n_trials=50)

    model = ExtraTreesClassifier()
    cv_results = calculating_cross_validation(model)
    metric_default_hyperparameters = [cv_results[f'test_{metric}'].mean() for metric in scoring.keys()]
    extra_trees_best_params = study.best_params
    
    mlflow.log_metric('average_precision_default_params', metric_default_hyperparameters[0])
    mlflow.log_metric('best_cv_average_precision', study.best_value)
    mlflow.log_params(study.best_params)
    
print(f'Best average precision score: {study.best_params}')
print(f'Best parameters: {study.best_params}')

# Comparison using best hyperparameters

In [ ]:
catboost_best_params['random_state'] = random_state
xgb_best_params['random_state'] = random_state
extra_trees_best_params['random_state'] = random_state
xgb_best_params["scale_pos_weight"] = scale_pos_weight

In [ ]:
catboost = CatBoostClassifier(**catboost_best_params)
xgboost = xgb.XGBClassifier(**xgb_best_params)
extra_tree = ExtraTreesClassifier(**extra_trees_best_params)


In [ ]:
candidate_models = {
                   'CatBoost': catboost,
                   'XgBoost': xgboost,
                   'Extra Trees': extra_tree
                    }

In [ ]:
#metrics that will be used for model comparison
metrics = ['Balanced Accuracy Score',
           'Precision Score',
           'Recall Score',
           'F1 Score',
           'Average Precision Score',
           'Roc AUC']


In [ ]:
scoring = {
    'balanced_accuracy': 'balanced_accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'average_precision': 'average_precision',
    'roc_auc': 'roc_auc'
}

In [ ]:
comparison_best_hyperparameters = pd.DataFrame(columns = metrics,
                       index= candidate_models,
                       data = 0.0)

In [ ]:
comparison_best_hyperparameters

# Logging to MLflow

In [ ]:
def log_artifacts(dataframe: pd.DataFrame, 
                  file_name: str, 
                  artifacts_path_mlflow: str) -> None:
    
    '''
    Log a dataframe to mlflow as an artifact and as a table.

    Args:
        dataframe (pd.DatsFrame): The pandas dataframe to be logged.
        file_name (str): name of the file
        artifacts_path_mlflow (str): folder to save the artifacts in mlflow
    
    '''

    df_log = dataframe.reset_index()

    with tempfile.TemporaryDirectory() as tmp_dir:
        file_path_parquet = Path(tmp_dir) / f"{file_name}.parquet"
        file_path_json = Path(tmp_dir) / f"{file_name}.json"

        # save parquet
        df_log.to_parquet(file_path_parquet)

        #save json
        #df_log.to_json(file_path_json, orient="split")
        
        
        # log dataset completo
        mlflow.log_artifact(
            str(file_path_parquet),
            artifact_path=artifacts_path_mlflow
        )

        # log preview visualizável no MLflow
        mlflow.log_table(
            df_log,
            artifact_file=f"{artifacts_path_mlflow}/{file_name}.json"
        )
        

In [ ]:
with mlflow.start_run(run_name="Model_comparison_using_best_hyperparameters", nested=True):
    for model_name, model in candidate_models.items():
        print(f'Training model {model_name}')
    
        cv_results = cross_validate(model, 
                                    x_train, 
                                    y_train, 
                                    cv=cv, 
                                    scoring=scoring, 
                                    verbose = False)
       
        metrics_results = [cv_results[f'test_{metric}'].mean() for metric in scoring.keys()]
        print(metrics_results)
        comparison_best_hyperparameters.loc[model_name, :] = metrics_results

    log_artifacts(comparison_best_hyperparameters,
                 'comparison_best_hyperparameters',
                 artifacts_path)
    '''
    comparison_best_hyperparameters.to_parquet(
        artifacts_dir / "model_selection_tuned_hyperparameters.parquet")
    mlflow.log_artifact(
        artifacts_dir / "model_selection_tuned_hyperparameters.parquet",
        artifact_path="model_selection")
    '''

In [ ]:
comparison_best_hyperparameters

In [ ]:
mlflow.end_run()